# Chapitre 12 - Exercices 12.6 à 12.11

* 12.6 : *optimal scoring* et problème aux valeurs propres ;
* 12.7 : *optimal scoring* pénalisé (PDA) ;
* 12.8 : les coefficients de l'*optimal scoring* sont proportionnels aux directions de la LDA ;
* 12.9 : LDA sur les valeurs ajustées d'une régression sur indicatrices ;
* 12.10 : LDA régularisée à noyau ;
* 12.11 : un modèle de mélange à centres partagés (MDA2) et son algorithme EM.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch12_svm import (LDA, FDA, MDA, class_indicator, second_diff_penalty, kmeans,
                          rbf_kernel, poly2_features, polynomial_kernel, simulate_mixture,
                          mixture_bayes_prob, plot_mixture_boundary, simulate_waveform)
set_style()

Notations communes : $Y$ est la matrice indicatrice $N\times K$ des classes,
$D_\pi = Y^TY/N = \mathrm{diag}(\hat\pi_1,\dots,\hat\pi_K)$ la matrice des proportions de
classes, $H$ la matrice $N\times M$ des $h_j(x_i)$. Nous utilisons les données *vowel*
(11 classes, 10 variables) pour les vérifications.

In [2]:
Xv, gv = datasets.load_vowel("train")
Yv, cls = class_indicator(gv)
N, K = Yv.shape
Dpi = Yv.T @ Yv / N
Hv = np.column_stack([np.ones(N), Xv])          # base : constante + variables
print("N = %d, K = %d, p = %d" % (N, K, Xv.shape[1]))

N = 528, K = 11, p = 10


## Exercice 12.6 : *Optimal scoring* et valeurs propres

**Énoncé (reformulé).** La régression de la FDA est un développement linéaire sur des
fonctions de base $h_m(x)$, $m = 1,\dots,M$.

(a) Montrer que le problème (12.52) s'écrit $\min_{\theta,\beta}\|Y\theta - H\beta\|^2$ (12.65),
où $\theta\in\mathbb R^K$.
(b) Interpréter les normalisations $\theta^TD_\pi\mathbf 1 = 0$ et $\theta^TD_\pi\theta = 1$.
(c) Montrer qu'après optimisation en $\beta$, on obtient $\max_\theta\theta^TY^TSY\theta$ (12.66)
sous ces contraintes, où $S$ est le projecteur associé à $H$.
(d) Si les $h_j$ contiennent la fonction constante, montrer que la plus grande valeur propre
de $S$ vaut 1.
(e) Avec une matrice $\Theta$ ($K\times K$) de scores normalisée par $\Theta^TD_\pi\Theta = I$,
montrer que la solution de (12.53) est donnée par l'ensemble des vecteurs propres ; le
premier est trivial et assure le centrage, les autres caractérisent la solution.

**(a)** Le score de l'observation $i$ est $\theta(g_i) = \sum_k y_{ik}\theta_k = (Y\theta)_i$
puisque la ligne $i$ de $Y$ ne contient qu'un 1, en colonne $g_i$. La prédiction est
$\sum_m\beta_mh_m(x_i) = (H\beta)_i$. Donc $\sum_i(\theta(g_i) - h(x_i)^T\beta)^2 = \|Y\theta - H\beta\|^2$.

**(b)** $\theta^TD_\pi\mathbf 1 = \sum_k\hat\pi_k\theta_k = \frac1N\sum_i\theta(g_i)$ : les
scores sont **centrés** sur l'échantillon. $\theta^TD_\pi\theta = \frac1N\sum_i\theta(g_i)^2$ :
avec le centrage, c'est la **variance** empirique des scores, fixée à 1. Ces contraintes
excluent la solution triviale $\theta = 0$ (et la solution constante).

**(c)** À $\theta$ fixé, (12.65) est un problème de moindres carrés en $\beta$ : $\hat\beta =
(H^TH)^{-1}H^TY\theta$ et $H\hat\beta = SY\theta$ avec $S = H(H^TH)^{-1}H^T$, projecteur
orthogonal ($S^T = S$, $S^2 = S$). Le résidu vaut

$$\|(I-S)Y\theta\|^2 = \theta^TY^T(I-S)^T(I-S)Y\theta = \theta^TY^T(I-S)Y\theta = N\theta^TD_\pi\theta - \theta^TY^TSY\theta = N - \theta^TY^TSY\theta,$$

en utilisant $(I-S)^T(I-S) = I - S$ et $Y^TY = ND_\pi$. Minimiser le résidu revient donc à
maximiser $\theta^TY^TSY\theta$ sous les contraintes de (b).

**(d)** $S$ est un projecteur : si $Sv = \mu v$ avec $v\ne0$, alors $\mu v = Sv = S^2v = \mu^2v$,
donc $\mu\in\{0, 1\}$. Si la constante est dans l'espace engendré par les colonnes de $H$,
alors $S\mathbf 1 = \mathbf 1$ : 1 est valeur propre, c'est la plus grande.

Version « problème d'optimal scoring » : le quotient $\frac{\theta^TY^TSY\theta}{N\theta^TD_\pi\theta} = \frac{\|SY\theta\|^2}{\|Y\theta\|^2}\le1$
(une projection orthogonale diminue la norme), et il vaut 1 pour $\theta = \mathbf 1$ car
$Y\mathbf 1 = \mathbf 1$ et $S\mathbf 1 = \mathbf 1$. La plus grande valeur propre généralisée de
$(\frac1NY^TSY, D_\pi)$ vaut donc 1, atteinte par les scores constants.

**(e)** Avec $L$ colonnes de scores, (12.53) se réduit de même à maximiser
$\mathrm{tr}(\Theta^TY^TSY\Theta)$ sous $\Theta^TD_\pi\Theta = I$. Posons $\Phi = D_\pi^{1/2}\Theta$ :
la contrainte devient $\Phi^T\Phi = I$ et l'objectif $\mathrm{tr}(\Phi^TA\Phi)$ avec
$A = D_\pi^{-1/2}Y^TSYD_\pi^{-1/2}$ symétrique. Le maximum d'une trace sous contrainte
d'orthonormalité est atteint par les vecteurs propres de $A$ associés aux plus grandes valeurs
propres (théorème de Ky Fan, généralisation du quotient de Rayleigh). Revenant à $\Theta$ : les
colonnes de $\Theta$ sont les vecteurs propres **généralisés**
$Y^TSY\theta = \mu\,ND_\pi\theta$.

* Le premier ($\mu = 1$) est $\theta = \mathbf 1$ (normalisé : $\mathbf 1^TD_\pi\mathbf 1 = 1$) :
  solution triviale.
* Les autres sont $D_\pi$-orthogonaux au premier : $\theta^TD_\pi\mathbf 1 = 0$, c'est-à-dire
  **centrés** - la contrainte (b) est automatiquement satisfaite. Ce sont les scores optimaux.

**Vérification numérique** sur *vowel* avec $H = [\mathbf 1, X]$.

In [ ]:
S = Hv @ np.linalg.solve(Hv.T @ Hv, Hv.T)
print("valeurs propres de S dans {0,1} :", np.allclose(np.sort(np.linalg.eigvalsh(S))[-(Hv.shape[1] + 1):].round(8),
                                                     np.r_[0, np.ones(Hv.shape[1])]))
mu, Theta = eigh(Yv.T @ S @ Yv / N, Dpi)
mu, Theta = mu[::-1], Theta[:, ::-1]
print("valeurs propres généralisées :", mu.round(4))
print("premier vecteur propre (constant ?) :", (Theta[:, 0] * np.sign(Theta[0, 0])).round(4))
print("Theta^T Dpi Theta = I :", np.allclose(Theta.T @ Dpi @ Theta, np.eye(K)))
print("scores suivants centrés (theta^T Dpi 1 = 0) :", np.allclose(Theta[:, 1:].T @ Dpi @ np.ones(K), 0))

Il y a $\min(K-1, p) = 10$ valeurs propres non triviales non nulles ; la dernière est nulle car
$p + 1 = 11 = K$. Vérifions enfin que le résidu moyen vaut bien $1 - \mu_\ell$ pour chaque score :

In [ ]:
for l in [1, 2, 3]:
    th = Theta[:, l]
    resid = Yv @ th - S @ Yv @ th
    print("score %d : ASR = %.4f   1 - mu = %.4f" % (l, resid @ resid / N, 1 - mu[l]))

> **À retenir.** L'*optimal scoring* est un problème aux valeurs propres généralisé
> $Y^TSY\theta = \mu ND_\pi\theta$ ; la solution constante ($\mu = 1$) absorbe le centrage.

## Exercice 12.7 : *Optimal scoring* pénalisé

**Énoncé.** Obtenir la solution du problème d'*optimal scoring* pénalisé (12.57) :

$$\min\ \frac1N\sum_{\ell}\left[\sum_i(\theta_\ell(g_i) - h(x_i)^T\beta_\ell)^2 + \lambda\beta_\ell^T\Omega\beta_\ell\right].$$

**Solution.** *Étape 1 : optimisation en $\beta$ à $\theta$ fixé.* Pour un score, le critère
$\|Y\theta - H\beta\|^2 + \lambda\beta^T\Omega\beta$ est quadratique convexe en $\beta$ ; son
gradient $-2H^T(Y\theta - H\beta) + 2\lambda\Omega\beta$ s'annule pour

$$\hat\beta = (H^TH + \lambda\Omega)^{-1}H^TY\theta\qquad\text{(régression ridge généralisée)}.$$

*Étape 2 : valeur du critère.* En notant $A = H^TH + \lambda\Omega$, on développe et on utilise
$A\hat\beta = H^TY\theta$ :

$$\|Y\theta - H\hat\beta\|^2 + \lambda\hat\beta^T\Omega\hat\beta = \theta^TY^TY\theta - 2\hat\beta^TH^TY\theta + \hat\beta^TA\hat\beta = \theta^TY^TY\theta - \theta^TY^TS_\lambda Y\theta,$$

avec $S_\lambda = H(H^TH + \lambda\Omega)^{-1}H^T$ (l'opérateur de lissage, qui n'est plus un
projecteur).

*Étape 3 : scores.* Avec la normalisation $\theta^TD_\pi\theta = 1$, minimiser revient à
maximiser $\theta^TY^TS_\lambda Y\theta$ : les scores optimaux sont les vecteurs propres
généralisés de $(\frac1NY^TS_\lambda Y, D_\pi)$, exactement comme en 12.6(e) avec $S_\lambda$ à
la place de $S$. Si la constante est dans la base et **non pénalisée**, $S_\lambda\mathbf 1 = \mathbf 1$
(le vecteur $\beta = e_1$ vérifie $A e_1 = H^T\mathbf 1$ car $\Omega e_1 = 0$) : la solution
triviale subsiste et les autres scores sont centrés. Enfin $\hat\beta_\ell = A^{-1}H^TY\theta_\ell$.

*Étape 4 : lien avec la PDA.* Séparons la constante et notons $\tilde H$ la matrice des
$h(x_i)$ centrés, $\Sigma_T = \tilde H^T\tilde H/N$, $M$ la matrice $K\times M$ des moyennes de
classes centrées, $\Sigma_B = M^TD_\pi M$ et $\Sigma_W = \Sigma_T - \Sigma_B$. Alors
$\tilde H^TY = NM^TD_\pi$ et $u = \hat\beta = (\Sigma_T + \frac\lambda N\Omega)^{-1}M^TD_\pi\theta$.
L'équation aux valeurs propres $D_\pi M u = \mu D_\pi\theta$ donne $Mu = \mu\theta$, d'où
$\Sigma_Bu = M^TD_\pi Mu = \mu M^TD_\pi\theta = \mu(\Sigma_T + \frac\lambda N\Omega)u$, puis
(avec $\Sigma_T = \Sigma_W + \Sigma_B$)

$$\Sigma_Bu = \frac{\mu}{1-\mu}\left(\Sigma_W + \tfrac\lambda N\Omega\right)u.$$

Les $\hat\beta_\ell$ sont donc les directions de la **PDA** : maximiser $u^T\Sigma_Bu$ sous
$u^T(\Sigma_W + \lambda'\Omega)u = 1$ avec $\lambda' = \lambda/N$ (cf. (12.58)). $\square$